# 個人記帳分析 MoneyBook

**資料庫管理**・教師示範專題　<a href="https://colab.research.google.com/github/chang-ye-tu/db/blob/master/demo/moneybook/moneybook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

課程首頁：[github.com/chang-ye-tu/db](https://github.com/chang-ye-tu/db)・大綱：[syllabus.md](https://github.com/chang-ye-tu/db/blob/master/syllabus.md)・專題：[projects.md](https://github.com/chang-ye-tu/db/blob/master/projects.md)

**第二個「做完了」的專題**——流水帳＋統計分析型系統的完成度標竿。（本題**不在**指派清單內：請模仿結構與完成度，別抄程式。）

> **怎麼使用**：同 library 示範——當標竿（結構＝檢核表）、當導讀（markdown 照著讀）、當零件庫（換成你的題目）。
> 這一本的看點跟 library 不同：**流水帳（append-only 事件流）＋快照對帳＋預算比對＋統計含金量高的報表**——
> 做「交易流水＋分析」類題目的同學請特別參考。
>
> 全本可重跑：`執行階段 → 全部執行`，約 1 分鐘。
> 本示範把「今天」固定為 `DEMO_TODAY = "2026-11-15"`；資料、預算、業務函數與 UI 共用同一日期，
> UI 可輸入其他 `YYYY-MM-DD` 日期，不會受真正執行日影響。

## 共同要求 10 條 ↔ 本 notebook 對照

| # | 要求 | 在哪一節 |
|---|---|---|
| 1 | ≥4 表 3NF＋約束＋schema 圖 | §1–§2 |
| 2 | 固定 seed 合成 ≥10,000 列 | §3 |
| 3 | CRUD＋`?` 傳值＋表單驗證 | §4、§8 |
| 4 | 交易保護＋併發競態示範 | §4、**§5** |
| 5 | 報表 ≥5（window／join／圖） | §6 |
| 6 | 最慢查詢 EXPLAIN＋索引前後計時 | §7 |
| 7 | Gradio ≥3 分頁 | §8 |
| 8 | ≥8 assert（含 2 個應該失敗） | 各節隨做隨測＋§9 總驗收 |
| 9 | AI 使用說明 | §10 |
| 10 | 簡報與 demo 腳本 | §11 |

# §1 情境、需求與 schema 設計（共同要求 1）

## 1.1 情境訪談稿

> 「我想把**每一筆收支**記下來：發生在哪個**帳戶**（現金、銀行、悠遊卡、信用卡）、
> 屬於哪個**分類**（餐飲、交通、房租、薪水⋯⋯）、金額、時間、備註。
> 帳戶之間會**轉帳**（例如從銀行幫悠遊卡儲值）——那不是收入也不是支出，別算進報表。
> 每個月我想給幾個分類訂**預算**，超支要看得到。
> 最想要的報表：每月結餘趨勢、錢都花去哪、預算達成率、我的大額支出有多誇張。」

## 1.2 名詞動詞分析

| 名詞 → 實體 | 動詞 → 關係 | 藏在句子裡的規則 |
|---|---|---|
| 帳戶 account | 「記在帳戶上」→ txn.account_id | 現金／票證不可為負；**信用卡可以**（欠款） |
| 分類 category（收入／支出／轉帳） | 「屬於分類」→ txn.cat_id | 轉帳不進收支報表 |
| 交易 txn（流水，**只插入不修改金額**） | 帳戶「互轉」→ 一次兩筆 txn | 轉出＋轉入**必須同時成立** |
| 預算 budget | 分類×月「訂預算」 | 同分類同月只能一筆（UPSERT 更新） |

## 1.3 ER 圖與設計決策

```
 account ──1───記───N── txn ──N───屬於───1── category
 (帳戶)               (交易流水)             (分類：收入/支出/轉帳)
                       category ──1───訂───N── budget（分類×月，UNIQUE）
```

三條**設計決策**（報告 Q&A 必問）：
1. **金額一律存正數**，方向由 `category.kind` 決定——避免「正負號自己記」的人為錯誤；
   報表用 `CASE kind` 換算正負（U02 的重編碼）。
2. **帳戶餘額＝快照欄**（記帳頻繁、餘額查詢更頻繁），每筆記帳的交易同步維護；
   §3 末與 §9 用「Σ(收) − Σ(支)」對帳稽核（U04 模式⑤）。
3. **跨欄位 CHECK**：`CHECK (kind = '信用卡' OR balance >= 0)`——只有信用卡能負，
   現金刷到負數是 bug 不是人生。

In [ ]:
#@title 📦 §2 建庫：DDL（4 表；NOT NULL／UNIQUE／CHECK／DEFAULT／跨欄 CHECK 全用上）
import sqlite3, os
from datetime import date
import numpy as np, pandas as pd

DEMO_TODAY = "2026-11-15"
DEMO_MONTH = DEMO_TODAY[:7]

def validate_iso_date(value, field="日期"):
    # 接受嚴格 YYYY-MM-DD；回傳正規化字串。
    text = str(value or "").strip()
    try:
        parsed = date.fromisoformat(text)
    except (TypeError, ValueError):
        raise ValueError(f"{field}請用 YYYY-MM-DD（例如 {DEMO_TODAY}）")
    if parsed.isoformat() != text:
        raise ValueError(f"{field}請用 YYYY-MM-DD（例如 {DEMO_TODAY}）")
    return text

def validate_iso_month(value, field="月份"):
    # 接受嚴格 YYYY-MM；budget 的自然鍵使用這個格式。
    text = str(value or "").strip()
    try:
        parsed = date.fromisoformat(f"{text}-01")
    except (TypeError, ValueError):
        raise ValueError(f"{field}請用 YYYY-MM（例如 {DEMO_MONTH}）")
    if len(text) != 7 or parsed.strftime("%Y-%m") != text:
        raise ValueError(f"{field}請用 YYYY-MM（例如 {DEMO_MONTH}）")
    return text

assert validate_iso_date(DEMO_TODAY) == DEMO_TODAY
assert validate_iso_month(DEMO_MONTH) == DEMO_MONTH

if os.path.exists("moneybook.db"):
    os.remove("moneybook.db")
mcon = sqlite3.connect("moneybook.db", check_same_thread=False)
mcon.row_factory = sqlite3.Row
mcon.executescript(f"""
PRAGMA foreign_keys = ON;

CREATE TABLE account(
  account_id INTEGER PRIMARY KEY,
  aname   TEXT NOT NULL UNIQUE,
  kind    TEXT NOT NULL CHECK (kind IN ('現金','銀行','電子票證','信用卡')),
  balance INTEGER NOT NULL DEFAULT 0,
  CHECK (kind = '信用卡' OR balance >= 0)          -- 設計決策 3：只有信用卡能負
);

CREATE TABLE category(
  cat_id INTEGER PRIMARY KEY,
  cname  TEXT NOT NULL UNIQUE,
  kind   TEXT NOT NULL CHECK (kind IN ('收入','支出','轉帳'))
);

CREATE TABLE txn(
  txn_id     INTEGER PRIMARY KEY,
  account_id INTEGER NOT NULL REFERENCES account(account_id),
  cat_id     INTEGER NOT NULL REFERENCES category(cat_id),
  amount     INTEGER NOT NULL CHECK (amount > 0),  -- 設計決策 1：一律正數
  at         TEXT NOT NULL DEFAULT ('{DEMO_TODAY} 12:00:00'),
  note       TEXT
);

CREATE TABLE budget(
  budget_id INTEGER PRIMARY KEY,
  cat_id    INTEGER NOT NULL REFERENCES category(cat_id),
  ym        TEXT NOT NULL,                          -- ISO 月份，例如 '2026-11'
  amount    INTEGER NOT NULL CHECK (amount > 0),
  UNIQUE (cat_id, ym)                               -- 同分類同月一筆（UPSERT 目標）
);
""")
mcon.executemany("INSERT INTO account(aname, kind, balance) VALUES (?,?,0)", [
    ("皮夾現金", "現金"), ("郵局帳戶", "銀行"), ("悠遊卡", "電子票證"), ("學生信用卡", "信用卡")])
mcon.executemany("INSERT INTO category(cname, kind) VALUES (?,?)", [
    ("薪資家教", "收入"), ("獎學金", "收入"), ("期初結轉", "收入"), ("利息", "收入"),
    ("早餐", "支出"), ("午餐", "支出"), ("晚餐", "支出"), ("飲料", "支出"), ("超商零食", "支出"),
    ("交通", "支出"), ("房租", "支出"), ("訂閱服務", "支出"), ("網購", "支出"),
    ("娛樂", "支出"), ("醫療", "支出"), ("電話費", "支出"),
    ("轉出", "轉帳"), ("轉入", "轉帳")])
mcon.commit()
print(f"moneybook.db 就緒 ✅ —— 模擬今日 {DEMO_TODAY}；4 表；帳戶 4 個、分類 18 個")

In [ ]:
# 約束踩點：每條規則踩一腳（之後也算進 §9「應該失敗」）
trials = [
    ("金額 0 元",   "INSERT INTO txn(account_id, cat_id, amount) VALUES (1, 5, 0)"),
    ("幽靈分類",    "INSERT INTO txn(account_id, cat_id, amount) VALUES (1, 999, 100)"),
    ("重複帳戶名",  "INSERT INTO account(aname, kind) VALUES ('皮夾現金', '現金')"),
    ("現金刷成負",  "UPDATE account SET balance = -50 WHERE aname = '皮夾現金'"),
    ("亂寫分類種類","INSERT INTO category(cname, kind) VALUES ('神秘', '其他')"),
]
for label, sql in trials:
    try:
        mcon.execute(sql); print(f"⚠️ {label}：竟然過了？！")
    except sqlite3.IntegrityError as e:
        print(f"✅ {label:10s} 擋下 → {e}")
mcon.rollback()

# §3 合成擬真資料（共同要求 2）：三年流水、10,000+ 列

**生成假設**（報告裡的「資料說明」）：**研究生情侶共同記帳**，截至 `DEMO_TODAY` 的近**三年**流水（頻率約單人 ×1.7）。

- **固定節奏**：每月 5 日入帳薪資家教、1 日扣房租、15 日扣訂閱與電話費；**3 日提款、10 日幫悠遊卡儲值**（轉帳，不進收支報表）；
- **日常長尾**：三餐幾乎天天記（金額 lognormal 右偏）、飲料與超商高頻小額、醫療低頻大額（右尾）；
- **時段與週節律**：交通集中平日、娛樂集中週末（各分類有自己的星期權重）；
- **通膨趨勢**：餐費逐年 +5%（時間趨勢——報表的移動平均會看到它）；
- **帳戶習慣**：房租薪資走銀行、小額走現金／悠遊卡、網購訂閱走信用卡（分類×帳戶的條件分佈）。
- **一致性**：每個帳戶 `balance ＝ Σ收入 − Σ支出 ± 轉帳`，合成完立刻對帳。

In [ ]:
#@title 🎲 合成 36 個月流水（固定 seed=7）：每個分類自己的頻率、金額分佈、星期權重
rng = np.random.default_rng(7)
N_MONTHS = 36
demo_month = np.datetime64(DEMO_MONTH, "M")
month_seq = [demo_month - np.timedelta64(N_MONTHS - 1, "M") + np.timedelta64(m, "M")
             for m in range(N_MONTHS)]                                      # 結束月永遠等於示範月

cat_id = {r["cname"]: r["cat_id"] for r in mcon.execute("SELECT cat_id, cname FROM category")}
acc_id = {r["aname"]: r["account_id"] for r in mcon.execute("SELECT account_id, aname FROM account")}

# (分類, [帳戶候選], 每月λ筆, 金額中位, 右偏σ, 星期權重[一..日], 時段小時池)——兩人份的頻率
SPECS = [
    ("早餐",    ["皮夾現金","悠遊卡"],     42, 55,   .25, [1,1,1,1,1,.7,.6],   [7,8,9]),
    ("午餐",    ["皮夾現金","悠遊卡"],     50, 110,  .30, [1,1,1,1,1,.9,.8],   [11,12,13]),
    ("晚餐",    ["皮夾現金","學生信用卡"], 46, 140,  .35, [1,1,1,1,1.2,1.3,1.1],[17,18,19,20]),
    ("飲料",    ["悠遊卡","皮夾現金"],     40, 60,   .25, [1,1,1,1,1.3,1.4,1], [10,14,15,16,21]),
    ("超商零食",["悠遊卡","皮夾現金"],     38, 45,   .35, [1]*7,               [8,12,16,21,22]),
    ("交通",    ["悠遊卡"],               70, 25,   .15, [1.2,1.2,1.2,1.2,1.2,.4,.3],[8,9,17,18]),
    ("娛樂",    ["學生信用卡","皮夾現金"], 10, 350,  .55, [.5,.5,.6,.7,1.4,2,1.8],[14,15,19,20,21]),
    ("網購",    ["學生信用卡"],            8, 520,  .65, [1]*7,               [12,13,21,22,23]),
    ("醫療",    ["皮夾現金"],              1, 400,  .80, [1,1,1,1,1,.8,.5],   [9,10,14]),
]
rows_txn = []
for cname, acc_pool, lam, med, sigma, week_w, hour_pool in SPECS:
    week_w = np.array(week_w, dtype=float)
    for m, mo in enumerate(month_seq):
        n = rng.poisson(lam)
        m_start = mo.astype("datetime64[D]")
        dse = m_start.astype(int)                             # 距 1970-01-01 的天數（那天是週四）
        day_pool = np.arange(28)
        w = week_w[(dse + day_pool + 3) % 7]                  # (＋3) 把週四對齊到索引 3
        days = rng.choice(day_pool, n, p=w / w.sum())
        trend = 1.05 ** (m / 12) if cname in ("早餐", "午餐", "晚餐", "飲料") else 1.0
        amts = np.maximum(np.round(rng.lognormal(np.log(med * trend), sigma, n)), 5).astype(int)
        for d, a in zip(days, amts):
            rows_txn.append((acc_id[rng.choice(acc_pool)], cat_id[cname], int(a),
                             f"{m_start + np.timedelta64(int(d), 'D')} {rng.choice(hour_pool):02d}:{rng.integers(60):02d}", None))
# 固定節奏：薪資（5 日）、房租（1 日）、訂閱＋電話（15 日）、每月提款與悠遊卡儲值（轉帳）
for m, mo in enumerate(month_seq):
    ym = str(mo)
    rows_txn.append((acc_id["郵局帳戶"], cat_id["薪資家教"],
                     int(rng.lognormal(np.log(41000), .10)), f"{ym}-05 09:10", "兩人薪資＋家教"))
    rows_txn.append((acc_id["郵局帳戶"], cat_id["房租"], 11000, f"{ym}-01 08:00", "套房月租"))
    rows_txn.append((acc_id["學生信用卡"], cat_id["訂閱服務"], 349, f"{ym}-15 00:05", "串流＋雲端"))
    rows_txn.append((acc_id["郵局帳戶"], cat_id["電話費"], 499, f"{ym}-15 10:00", None))
    rows_txn.append((acc_id["郵局帳戶"], cat_id["轉出"], 14000, f"{ym}-03 12:00", "ATM 提款"))
    rows_txn.append((acc_id["皮夾現金"], cat_id["轉入"], 14000, f"{ym}-03 12:00", "ATM 提款"))
    rows_txn.append((acc_id["郵局帳戶"], cat_id["轉出"], 9000, f"{ym}-10 12:30", "悠遊卡儲值"))
    rows_txn.append((acc_id["悠遊卡"],   cat_id["轉入"], 9000, f"{ym}-10 12:30", "悠遊卡儲值"))
    if m % 6 == 5:
        rows_txn.append((acc_id["郵局帳戶"], cat_id["獎學金"], 10000, f"{ym}-20 12:00", "系獎學金"))
first_month = str(month_seq[0])
rows_txn.append((acc_id["皮夾現金"], cat_id["期初結轉"], 12000, f"{first_month}-01 00:01", "期初"))
rows_txn.append((acc_id["郵局帳戶"], cat_id["期初結轉"], 90000, f"{first_month}-01 00:01", "期初"))
rows_txn.append((acc_id["悠遊卡"],   cat_id["期初結轉"], 8000,  f"{first_month}-01 00:01", "期初"))

# 示範月只保留 DEMO_TODAY 當天以前的資料；固定 seed 下仍完全可重現。
rows_txn = [row for row in rows_txn if row[3][:10] <= DEMO_TODAY]

with mcon:
    mcon.executemany("INSERT INTO txn(account_id, cat_id, amount, at, note) VALUES (?,?,?,?,?)", rows_txn)
print(f"txn {mcon.execute('SELECT COUNT(*) FROM txn').fetchone()[0]:,} 列 ✅（含每月兩筆例行轉帳）")

In [ ]:
#@title 🎲 預算（近 24 個月 × 8 個分類）＋把帳戶餘額快照「結算」出來
BUDGET_CATS = ["早餐","午餐","晚餐","飲料","交通","娛樂","網購","超商零食"]
BUDGET_BASE = {"早餐":2600,"午餐":6000,"晚餐":7200,"飲料":2600,"交通":1900,"娛樂":4000,"網購":4500,"超商零食":1900}
b_rows = []
for mo in month_seq[12:]:                                # 近 24 個月才開始記預算
    for c in BUDGET_CATS:
        b_rows.append((cat_id[c], str(mo), int(BUDGET_BASE[c] * rng.uniform(0.9, 1.15))))
with mcon:
    mcon.executemany("INSERT INTO budget(cat_id, ym, amount) VALUES (?,?,?)", b_rows)

# 帳戶餘額快照 = Σ收 − Σ支 ± 轉帳（轉入＋、轉出−）——「事實在流水、快照可重算」
with mcon:
    mcon.execute("""
        UPDATE account SET balance = COALESCE((
            SELECT SUM(CASE c.kind
                         WHEN '收入' THEN t.amount
                         WHEN '支出' THEN -t.amount
                         ELSE CASE WHEN c.cname = '轉入' THEN t.amount ELSE -t.amount END END)
            FROM txn t JOIN category c ON t.cat_id = c.cat_id
            WHERE t.account_id = account.account_id), 0)""")
print(pd.read_sql_query("SELECT aname 帳戶, kind 種類, balance 餘額 FROM account", mcon).to_string(index=False))

total_rows = sum(mcon.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
                 for t in ["account", "category", "txn", "budget"])
print(f"\n合計 {total_rows:,} 列", "✅（共同要求 2 達標）" if total_rows >= 10_000 else "❌ 不足一萬列")
assert total_rows >= 10_000
for acc_name in ("皮夾現金", "悠遊卡"):
    b = mcon.execute("SELECT balance FROM account WHERE aname = ?", (acc_name,)).fetchone()[0]
    assert b >= 0, f"{acc_name} 不得為負（每月提款/儲值的節奏要夠）"
print("✅ 現金／票證餘額非負；信用卡餘額為負＝欠款（跨欄 CHECK 的設計意圖）")

In [ ]:
#@title 🈶 圖表中文字型（Colab 需安裝一次；本機有 Noto 就直接生效）
import os, glob, subprocess, matplotlib
from matplotlib import font_manager

cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if not cjk:                                    # Colab 第一次：裝字型（約 10 秒）
    subprocess.run(["apt-get", "-qq", "install", "-y", "fonts-noto-cjk"], capture_output=True)
    cjk = sorted(glob.glob("/usr/share/fonts/opentype/noto/NotoSansCJK*.ttc"))
if cjk:
    font_manager.fontManager.addfont(cjk[0])
    matplotlib.rcParams["font.family"] = font_manager.FontProperties(fname=cjk[0]).get_name()
    print("中文字型就緒 ✅：", os.path.basename(cjk[0]))
else:
    print("⚠️ 找不到 CJK 字型——圖表中文會變 □（不影響其他功能）")
matplotlib.rcParams["axes.unicode_minus"] = False

In [ ]:
# 分佈自我檢查：金額右偏＋月支出節律（報告放這兩張，統計系的品味在這裡亮出來）
import matplotlib.pyplot as plt

df = pd.read_sql_query("""SELECT t.amount, substr(t.at, 1, 7) AS ym, c.kind
                          FROM txn t JOIN category c ON t.cat_id = c.cat_id""", mcon)
exp = df[df.kind == "支出"]
fig, axes = plt.subplots(1, 2, figsize=(11, 3))
axes[0].hist(exp.amount, bins=60)
axes[0].set_title("支出金額分佈（右偏）"); axes[0].set_xlabel("元")
exp.groupby("ym").amount.sum().plot(ax=axes[1], marker="o", ms=2.5)
axes[1].set_title("每月總支出"); axes[1].tick_params(axis='x', rotation=60, labelsize=7)
plt.tight_layout(); plt.show()
print(f"支出中位數 {exp.amount.median():.0f} vs 平均 {exp.amount.mean():.0f} → 右偏 ✅；"
      f"月支出隨年微升 → 通膨趨勢 ✅")

# §4 資料層：CRUD 函數（共同要求 3、4）

| 類 | 函數 | 交易？ |
|---|---|---|
| C | `add_txn`（**核心操作①**：寫流水＋更新餘額快照） `set_budget`（UPSERT） | ✔ |
| R | `list_txns` `budget_status` | — |
| U | `transfer`（**核心操作②**：一次兩帳戶兩筆流水） | ✔ |
| D | `delete_txn`（連同回沖餘額——刪除也要交易！） | ✔ |

In [ ]:
# C（核心操作①）：記一筆——「寫流水 ＋ 動餘額快照」同生共死
def add_txn(acc_name, cname, amount, note=None, at=DEMO_TODAY):
    try:
        amount = int(amount)
        if amount <= 0:
            return "⚠️ 金額要是正數（方向由分類決定，別自己加負號）"
        at = validate_iso_date(DEMO_TODAY if at is None else at, "交易日期")
        at_value = f"{at} 12:00:00"
        with mcon:
            acc = mcon.execute("SELECT account_id FROM account WHERE aname = ?", (acc_name,)).fetchone()
            cat = mcon.execute("SELECT cat_id, kind FROM category WHERE cname = ?", (cname,)).fetchone()
            if acc is None or cat is None:
                raise ValueError("帳戶或分類不存在")
            if cat["kind"] == "轉帳":
                raise ValueError("轉帳請用「轉帳」功能（要同時動兩個帳戶）")
            sign = 1 if cat["kind"] == "收入" else -1
            mcon.execute("""INSERT INTO txn(account_id, cat_id, amount, at, note)
                            VALUES (?,?,?,?,?)""",
                         (acc["account_id"], cat["cat_id"], amount, at_value, note))
            mcon.execute("UPDATE account SET balance = balance + ? WHERE account_id = ?",
                         (sign * amount, acc["account_id"]))    # DB 端算術：原子（§5 解釋為什麼）
        return f"✅ 已記：{cname} {sign * amount:+,} 元（{acc_name}）"
    except ValueError as e:
        return f"❌ {e}"
    except sqlite3.IntegrityError:
        return "❌ 這筆會讓帳戶變負（只有信用卡可以透支）——整筆取消"

print(add_txn("皮夾現金", "晚餐", 120, "自助餐"))
print(add_txn("皮夾現金", "晚餐", -50))                        # 應該失敗：負數
print(add_txn("皮夾現金", "醫療", 10**9))                      # 應該失敗：現金刷爆（跨欄 CHECK 擋）
assert add_txn("沒這帳戶", "晚餐", 100).startswith("❌")
print("✅ add_txn OK（成功／負數／刷爆／幽靈帳戶四情境）")

In [ ]:
# U（核心操作②）：轉帳——兩個帳戶、兩筆流水，四個動作一個交易
def transfer(src, dst, amount, note=None, at=DEMO_TODAY):
    try:
        amount = int(amount)
        if amount <= 0 or src == dst:
            return "⚠️ 金額要正、兩個帳戶要不同"
        at = validate_iso_date(DEMO_TODAY if at is None else at, "轉帳日期")
        at_value = f"{at} 12:00:00"
        with mcon:
            a = mcon.execute("SELECT account_id FROM account WHERE aname = ?", (src,)).fetchone()
            b = mcon.execute("SELECT account_id FROM account WHERE aname = ?", (dst,)).fetchone()
            if a is None or b is None:
                raise ValueError("帳戶不存在")
            note2 = note or f"{src} → {dst}"
            mcon.execute("""INSERT INTO txn(account_id, cat_id, amount, at, note)
                            VALUES (?, ?, ?, ?, ?)""",
                         (a["account_id"], cat_id["轉出"], amount, at_value, note2))
            mcon.execute("""INSERT INTO txn(account_id, cat_id, amount, at, note)
                            VALUES (?, ?, ?, ?, ?)""",
                         (b["account_id"], cat_id["轉入"], amount, at_value, note2))
            mcon.execute("UPDATE account SET balance = balance - ? WHERE account_id = ?",
                         (amount, a["account_id"]))
            mcon.execute("UPDATE account SET balance = balance + ? WHERE account_id = ?",
                         (amount, b["account_id"]))
        return f"✅ 轉帳完成：{src} → {dst} {amount:,} 元"
    except ValueError as e:
        return f"❌ {e}"
    except sqlite3.IntegrityError:
        return "❌ 餘額不足以轉出（整筆取消：兩邊都沒動）"

print(transfer("郵局帳戶", "悠遊卡", 1000, "悠遊卡儲值"))
print(transfer("皮夾現金", "悠遊卡", 10**9))                   # 應該失敗：現金不夠 → 四個動作全回滾
card_bal = mcon.execute("SELECT balance FROM account WHERE aname='悠遊卡'").fetchone()[0]
print("悠遊卡餘額：", card_bal)
assert transfer("郵局帳戶", "郵局帳戶", 100).startswith("⚠️")
print("✅ transfer OK（成功／不足／同帳戶三情境；失敗時兩邊都沒動＝原子性）")

In [ ]:
# C：設預算（UPSERT：同分類同月已有就改金額——U02 的 ON CONFLICT DO UPDATE 實戰）
def set_budget(cname, ym, amount):
    try:
        amount = int(amount)
        ym = validate_iso_month(ym, "預算月份")
        cat = mcon.execute("SELECT cat_id, kind FROM category WHERE cname = ?", (cname,)).fetchone()
        if cat is None or cat["kind"] != "支出":
            return "⚠️ 只能對「支出」分類設預算"
        with mcon:
            mcon.execute("""INSERT INTO budget(cat_id, ym, amount) VALUES (?,?,?)
                            ON CONFLICT(cat_id, ym) DO UPDATE SET amount = excluded.amount""",
                         (cat["cat_id"], ym, amount))
        return f"✅ {ym} {cname} 預算 = {amount:,}"
    except ValueError as e:
        return f"⚠️ {e}"
    except sqlite3.IntegrityError as e:
        return f"❌ {e}"

print(set_budget("娛樂", DEMO_MONTH, 3000))
print(set_budget("娛樂", DEMO_MONTH, 2600), "← 再設一次＝更新，不報錯")
n = mcon.execute("SELECT COUNT(*) FROM budget WHERE ym=? AND cat_id=?",
                 (DEMO_MONTH, cat_id["娛樂"])).fetchone()[0]
assert n == 1 and set_budget("薪資家教", DEMO_MONTH, 1).startswith("⚠️")
print("✅ set_budget OK（UPSERT 一筆到底；收入分類被擋）")

In [ ]:
# R＋D：查流水（join ×2）與刪一筆（刪除也要「回沖餘額」——一樣包交易）
def list_txns(acc_name="全部", keyword="", limit=15):
    sql = """SELECT t.txn_id AS 編號, t.at AS 時間, a.aname AS 帳戶, c.cname AS 分類,
                    CASE c.kind WHEN '收入' THEN t.amount ELSE -t.amount END AS 金額,
                    COALESCE(t.note, '') AS 備註
             FROM txn t JOIN account a ON t.account_id = a.account_id
                        JOIN category c ON t.cat_id = c.cat_id
             WHERE COALESCE(t.note, '') || c.cname LIKE ?"""
    params = [f"%{keyword}%"]
    if acc_name != "全部":
        sql += " AND a.aname = ?"; params.append(acc_name)
    sql += " ORDER BY t.at DESC LIMIT ?"; params.append(limit)
    return pd.read_sql_query(sql, mcon, params=params)

def delete_txn(txn_id):
    try:
        with mcon:
            t = mcon.execute("""SELECT t.account_id, t.amount, c.kind FROM txn t
                                JOIN category c ON t.cat_id = c.cat_id
                                WHERE t.txn_id = ?""", (txn_id,)).fetchone()
            if t is None:
                raise ValueError("查無此筆")
            if t["kind"] == "轉帳":
                raise ValueError("轉帳要成對刪，請用轉帳沖銷（示範從簡：擋下）")
            refund = -t["amount"] if t["kind"] == "收入" else t["amount"]
            mcon.execute("UPDATE account SET balance = balance + ? WHERE account_id = ?",
                         (refund, t["account_id"]))
            mcon.execute("DELETE FROM txn WHERE txn_id = ?", (txn_id,))
        return "🗑 已刪除並回沖餘額"
    except ValueError as e:
        return f"❌ {e}"

print(list_txns(limit=5).to_string(index=False))
last_id = mcon.execute("SELECT txn_id FROM txn ORDER BY txn_id DESC LIMIT 1").fetchone()[0]
bal0 = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
print(add_txn("皮夾現金", "飲料", 60)); print(delete_txn(last_id + 1))
bal1 = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
assert bal0 == bal1 and delete_txn(99999999).startswith("❌")
print("✅ delete_txn OK（記→刪→餘額復原；幽靈編號被擋）")

# §5 併發競態：兩支手機同時記帳（共同要求 4 的重頭戲）

> 【導讀】library 示範演的是**超賣**；這裡演另一種經典事故——**lost update**。
> 兩支手機（兩條連線）同時記帳，壞寫法是「先把餘額讀出來、Python 算好新值、再寫回去」。看看會怎樣。

In [ ]:
# 事故重現：read－modify－write——後寫的人把先寫的人蓋掉（lost update）
mcon.execute("UPDATE account SET balance = 1000 WHERE aname = '皮夾現金'"); mcon.commit()
con_a = sqlite3.connect("moneybook.db"); con_b = sqlite3.connect("moneybook.db")

a_reads = con_a.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
b_reads = con_b.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
print(f"甲讀到 {a_reads}，在手機上算好 {a_reads} − 120（晚餐）＝ {a_reads - 120}")
print(f"乙讀到 {b_reads}，在手機上算好 {b_reads} − 80（飲料）＝ {b_reads - 80}")

con_a.execute("UPDATE account SET balance = ? WHERE aname='皮夾現金'", (a_reads - 120,)); con_a.commit()
con_b.execute("UPDATE account SET balance = ? WHERE aname='皮夾現金'", (b_reads - 80,));  con_b.commit()
final_bal = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
print(f"\n兩筆都記完，餘額 = {final_bal}（正確答案是 1000 − 120 − 80 = 800）")
assert final_bal == 920
print("→ 甲扣的 120 元被乙「用舊值算的新值」整個蓋掉——沒有人報錯，錢就是不對。")
con_a.close(); con_b.close()

In [ ]:
# 防護：把算術交給資料庫——balance = balance − ?（單一 UPDATE 語句是原子的）
mcon.execute("UPDATE account SET balance = 1000 WHERE aname = '皮夾現金'"); mcon.commit()
con_a = sqlite3.connect("moneybook.db"); con_b = sqlite3.connect("moneybook.db")

con_a.execute("UPDATE account SET balance = balance - 120 WHERE aname='皮夾現金'"); con_a.commit()
con_b.execute("UPDATE account SET balance = balance - 80  WHERE aname='皮夾現金'"); con_b.commit()
final_bal = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
print(f"餘額 = {final_bal}（1000 − 120 − 80 ✅）")
assert final_bal == 800
con_a.close(); con_b.close()
print()
print("→ §4 的 add_txn() 從頭就是這樣寫的：讀值算值的工作留在 SQL 裡，兩人同時記帳也不互相蓋。")
print("  （搭配 CHECK 當最後防線；再進一階的 BEGIN IMMEDIATE 鎖法，U06 課堂拆解。）")
# 收尾：把示範用的餘額改回「由流水結算」的正確值——順便又用了一次對帳邏輯
with mcon:
    mcon.execute("""UPDATE account SET balance = COALESCE((
        SELECT SUM(CASE c.kind WHEN '收入' THEN t.amount WHEN '支出' THEN -t.amount
                   WHEN '轉帳' THEN CASE WHEN c.cname='轉入' THEN t.amount ELSE -t.amount END END)
        FROM txn t JOIN category c ON t.cat_id = c.cat_id
        WHERE t.account_id = account.account_id), 0)""")
print("（已把餘額快照重新結算回真實值）")

# §6 統計報表 ×6（共同要求 5：≥1 window、≥2 join、≥1 圖表）

轉帳（轉出／轉入）**一律排除**在收支報表外——設計決策 1 的紅利：一個 `WHERE kind != '轉帳'` 搞定。

In [ ]:
# 報表 1【每月收支與結餘？】收支樞紐＋累積結餘（window 累積）——本 App 的門面
pd.read_sql_query("""
    WITH m AS (SELECT substr(t.at, 1, 7) AS ym,
                      SUM(CASE WHEN c.kind = '收入' THEN t.amount ELSE 0 END) AS 收入,
                      SUM(CASE WHEN c.kind = '支出' THEN t.amount ELSE 0 END) AS 支出
               FROM txn t JOIN category c ON t.cat_id = c.cat_id
               WHERE c.kind != '轉帳'
               GROUP BY ym)
    SELECT ym AS 月份, 收入, 支出, 收入 - 支出 AS 結餘,
           SUM(收入 - 支出) OVER (ORDER BY ym) AS 累積結餘
    FROM m ORDER BY ym DESC LIMIT 8""", mcon)

In [ ]:
# 報表 2【錢都花去哪？】近 12 個月分類佔比（window 佔比）——圓餅圖的資料底
pd.read_sql_query("""
    WITH s AS (SELECT c.cname AS 分類, SUM(t.amount) AS 小計
               FROM txn t JOIN category c ON t.cat_id = c.cat_id
               WHERE c.kind = '支出' AND t.at >= date(?, '-12 month')
               GROUP BY c.cat_id)
    SELECT 分類, 小計, ROUND(小計 * 100.0 / SUM(小計) OVER (), 1) AS 佔比pct
    FROM s ORDER BY 小計 DESC""", mcon, params=(DEMO_TODAY,))

In [ ]:
# 報表 3【餐費有在漲嗎？】月餐飲支出＋3 月移動平均（window frame）——看得到通膨趨勢
pd.read_sql_query("""
    WITH m AS (SELECT substr(t.at, 1, 7) AS ym, SUM(t.amount) AS 餐飲
               FROM txn t JOIN category c ON t.cat_id = c.cat_id
               WHERE c.cname IN ('早餐','午餐','晚餐')
               GROUP BY ym)
    SELECT ym AS 月份, 餐飲,
           ROUND(AVG(餐飲) OVER (ORDER BY ym ROWS BETWEEN 2 PRECEDING AND CURRENT ROW)) AS 近三月平均
    FROM m ORDER BY ym DESC LIMIT 8""", mcon)

In [ ]:
# 報表 4【預算守住了嗎？】本月預算 vs 實際（join budget ＋ CASE 警示）——demo 亮點之一
def budget_status(ym):
    ym = validate_iso_month(ym, "報表月份")
    return pd.read_sql_query("""
        SELECT c.cname AS 分類, b.amount AS 預算, COALESCE(s.actual, 0) AS 實際,
               b.amount - COALESCE(s.actual, 0) AS 剩餘,
               CASE WHEN COALESCE(s.actual, 0) > b.amount THEN '🔥 超支'
                    WHEN COALESCE(s.actual, 0) > b.amount * 0.8 THEN '⚠️ 逼近'
                    ELSE 'OK' END AS 狀態
        FROM budget b
        JOIN category c ON b.cat_id = c.cat_id
        LEFT JOIN (SELECT cat_id, SUM(amount) AS actual FROM txn
                   WHERE substr(at, 1, 7) = ? GROUP BY cat_id) s ON s.cat_id = b.cat_id
        WHERE b.ym = ?
        ORDER BY (COALESCE(s.actual,0) * 1.0 / b.amount) DESC""", mcon, params=(ym, ym))

budget_status(DEMO_MONTH)

In [ ]:
# 報表 5【什麼時候在花錢？】星期 × 時段 熱點（條件式聚合樞紐）——行為的節律現形
pd.read_sql_query("""
    SELECT CASE strftime('%w', at) WHEN '0' THEN '日' WHEN '1' THEN '一' WHEN '2' THEN '二'
                WHEN '3' THEN '三' WHEN '4' THEN '四' WHEN '5' THEN '五' ELSE '六' END AS 星期,
           SUM(CASE WHEN CAST(strftime('%H', at) AS INT) BETWEEN 6  AND 10 THEN 1 ELSE 0 END) AS 早,
           SUM(CASE WHEN CAST(strftime('%H', at) AS INT) BETWEEN 11 AND 14 THEN 1 ELSE 0 END) AS 午,
           SUM(CASE WHEN CAST(strftime('%H', at) AS INT) BETWEEN 17 AND 20 THEN 1 ELSE 0 END) AS 晚,
           SUM(CASE WHEN CAST(strftime('%H', at) AS INT) >= 21 THEN 1 ELSE 0 END)             AS 深夜,
           COUNT(*) AS 全日
    FROM txn t JOIN category c ON t.cat_id = c.cat_id
    WHERE c.kind = '支出'
    GROUP BY strftime('%w', at)
    ORDER BY CASE strftime('%w', at) WHEN '0' THEN 7 ELSE CAST(strftime('%w', at) AS INT) END""", mcon)

In [ ]:
# 報表 6【我的大額支出多誇張？】Top 10 ＋ PERCENT_RANK（「這筆贏過 99.9% 的日常」）
pd.read_sql_query("""
    WITH e AS (SELECT t.at, c.cname, t.amount,
                      ROUND(PERCENT_RANK() OVER (ORDER BY t.amount) * 100, 2) AS 贏過pct
               FROM txn t JOIN category c ON t.cat_id = c.cat_id
               WHERE c.kind = '支出')
    SELECT at AS 時間, cname AS 分類, amount AS 金額, 贏過pct
    FROM e ORDER BY 金額 DESC LIMIT 10""", mcon)

In [ ]:
# 圖表版（等下嵌 gr.Plot）：月結餘趨勢＋近 12 月分類佔比
def net_chart():
    df = pd.read_sql_query("""
        WITH m AS (SELECT substr(t.at,1,7) ym,
                          SUM(CASE WHEN c.kind='收入' THEN t.amount ELSE 0 END) -
                          SUM(CASE WHEN c.kind='支出' THEN t.amount ELSE 0 END) AS net
                   FROM txn t JOIN category c ON t.cat_id = c.cat_id
                   WHERE c.kind != '轉帳' GROUP BY ym)
        SELECT ym, net, SUM(net) OVER (ORDER BY ym) AS cum FROM m""", mcon)
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.bar(df.ym, df.net, alpha=.4, label="單月結餘")
    ax.plot(df.ym, df.cum, lw=2, color="C1", label="累積結餘")
    ax.legend(); ax.set_title("每月結餘與累積"); ax.tick_params(axis='x', rotation=60, labelsize=6)
    plt.tight_layout(); return fig

def share_chart(as_of=DEMO_TODAY):
    as_of = validate_iso_date(as_of, "圖表基準日")
    df = pd.read_sql_query("""
        SELECT c.cname, SUM(t.amount) s FROM txn t JOIN category c ON t.cat_id=c.cat_id
        WHERE c.kind='支出' AND t.at >= date(?,'-12 month')
        GROUP BY c.cat_id ORDER BY s DESC LIMIT 9""", mcon, params=(as_of,))
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.barh(df.cname[::-1], df.s[::-1])
    ax.set_title("近 12 個月各分類支出")
    plt.tight_layout(); return fig

for f in (net_chart, share_chart):
    fig = f(); print(f.__name__, "→", type(fig).__name__); plt.close(fig)
print("✅ 報表 1–6 齊：window ×3（累積、移動平均、PERCENT_RANK）、join 全上、圖 ×2 —— 共同要求 5 達標")

# §7 效能：EXPLAIN → 索引 → 前後計時（共同要求 6）

最頻繁的查詢是「某分類最近的流水」（報表 4 每個月每個分類都要算一次）。
txn 一萬多列，先讓查詢計畫招供，再建**複合索引** `(cat_id, at)`。

In [ ]:
import time
q_sql = "SELECT * FROM txn WHERE cat_id = ? AND at >= date(?, 'start of year')"

print("── 建索引前 ──")
for r in mcon.execute(f"EXPLAIN QUERY PLAN {q_sql}", (5, DEMO_TODAY)):
    print("  ", r[3])
t = time.perf_counter()
for i in range(2000):
    mcon.execute(q_sql, (i % 16 + 1, DEMO_TODAY)).fetchall()
t_slow = time.perf_counter() - t

mcon.execute("CREATE INDEX IF NOT EXISTS idx_txn_cat_at ON txn(cat_id, at)")

print("\n── 建 idx_txn_cat_at 之後 ──")
for r in mcon.execute(f"EXPLAIN QUERY PLAN {q_sql}", (5, DEMO_TODAY)):
    print("  ", r[3])
t = time.perf_counter()
for i in range(2000):
    mcon.execute(q_sql, (i % 16 + 1, DEMO_TODAY)).fetchall()
t_fast = time.perf_counter() - t

print(f"\n2,000 次查詢：{t_slow*1000:.0f} ms → {t_fast*1000:.0f} ms（快 {t_slow/t_fast:.0f} 倍）")
assert t_fast < t_slow
print("✅ 共同要求 6 達標。複合索引 (cat_id, at)：先等值再範圍——最左前綴原理 U07 拆解")

# §8 Gradio 三分頁介面（共同要求 7）

| 分頁 | 內容 |
|---|---|
| ✏️ 記帳 | 記一筆（帳戶／分類下拉吃資料庫）＋最近流水即時刷新 |
| 🔁 轉帳與預算 | 帳戶互轉、設預算（UPSERT）＋帳戶餘額表 |
| 📊 報表 | 月份下拉 → 預算達成表；兩張圖 |

In [ ]:
import sys, importlib.util, subprocess
if importlib.util.find_spec("gradio") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"])
import gradio as gr

ACCOUNTS = [r["aname"] for r in mcon.execute("SELECT aname FROM account ORDER BY account_id")]
EXP_CATS = [r["cname"] for r in mcon.execute("SELECT cname FROM category WHERE kind='支出' ORDER BY cat_id")]
IN_EXP_CATS = [r["cname"] for r in mcon.execute(
    "SELECT cname FROM category WHERE kind IN ('收入','支出') ORDER BY kind DESC, cat_id")]
MONTHS = [r[0] for r in mcon.execute("SELECT DISTINCT ym FROM budget ORDER BY ym DESC")]

def balance_table():
    return pd.read_sql_query("SELECT aname 帳戶, kind 種類, balance 餘額 FROM account", mcon)

def ui_add_txn(acc, cat, amount, note, on_date):
    result = add_txn(acc, cat, amount, note or None, on_date)
    try:
        ym = validate_iso_date(on_date, "交易日期")[:7]
    except ValueError:
        ym = DEMO_MONTH
    return result, list_txns(limit=10), balance_table(), budget_status(ym)

def ui_transfer(src, dst, amount, on_date):
    return transfer(src, dst, amount, at=on_date), balance_table()

def ui_set_budget(cat, ym, amount):
    return set_budget(cat, ym, amount), budget_status(ym)

print("UI 包裝就緒；分類下拉", len(IN_EXP_CATS), "項、月份下拉", len(MONTHS), "項——全部從資料庫撈")

In [ ]:
with gr.Blocks(title="MoneyBook") as app:
    gr.Markdown("# 💰 MoneyBook 個人記帳分析（教師示範專題）")
    gr.Markdown(f"**可重現的模擬今日：`{DEMO_TODAY}`**。日期欄一律輸入 `YYYY-MM-DD`；不讀取執行環境的真正今天。")
    with gr.Tab("✏️ 記帳"):
        with gr.Row():
            dd_acc = gr.Dropdown(ACCOUNTS, value="皮夾現金", label="帳戶")
            dd_cat = gr.Dropdown(IN_EXP_CATS, value="午餐", label="分類（收入／支出）")
            num_amt = gr.Number(label="金額（正數）", value=100)
            txt_note = gr.Textbox(label="備註（選填）")
            txt_date = gr.Textbox(value=DEMO_TODAY, label="交易日期（YYYY-MM-DD）")
        msg_add = gr.Textbox(interactive=False, label="結果")
        with gr.Row():
            tbl_txns = gr.Dataframe(value=list_txns(limit=10), label="最近 10 筆")
            tbl_bal = gr.Dataframe(value=balance_table(), label="帳戶餘額")
        tbl_month_budget = gr.Dataframe(value=budget_status(DEMO_MONTH), label="交易月份預算即時連動")
        gr.Button("記下來", variant="primary").click(
            ui_add_txn, [dd_acc, dd_cat, num_amt, txt_note, txt_date],
            [msg_add, tbl_txns, tbl_bal, tbl_month_budget])
    with gr.Tab("🔁 轉帳與預算"):
        gr.Markdown("#### 帳戶互轉（例：幫悠遊卡儲值）")
        with gr.Row():
            dd_src = gr.Dropdown(ACCOUNTS, value="郵局帳戶", label="從")
            dd_dst = gr.Dropdown(ACCOUNTS, value="悠遊卡", label="到")
            num_tr = gr.Number(label="金額", value=1000)
            txt_tr_date = gr.Textbox(value=DEMO_TODAY, label="轉帳日期（YYYY-MM-DD）")
        msg_tr = gr.Textbox(interactive=False, label="結果")
        tbl_bal2 = gr.Dataframe(value=balance_table(), label="帳戶餘額")
        gr.Button("轉帳", variant="primary").click(
            ui_transfer, [dd_src, dd_dst, num_tr, txt_tr_date], [msg_tr, tbl_bal2])
        gr.Markdown("#### 設定月預算（同分類同月再設一次＝更新）")
        with gr.Row():
            dd_bcat = gr.Dropdown(EXP_CATS, value="娛樂", label="分類")
            dd_bym = gr.Dropdown(MONTHS, value=DEMO_MONTH, label="月份")
            num_bud = gr.Number(label="預算金額", value=3000)
        msg_bud = gr.Textbox(interactive=False, label="結果")
        tbl_bud = gr.Dataframe(value=budget_status(DEMO_MONTH), label="該月預算 vs 實際")
        gr.Button("設定").click(ui_set_budget, [dd_bcat, dd_bym, num_bud], [msg_bud, tbl_bud])
    with gr.Tab("📊 報表"):
        dd_rym = gr.Dropdown(MONTHS, value=DEMO_MONTH, label="看哪個月的預算達成")
        tbl_rep = gr.Dataframe(value=budget_status(DEMO_MONTH))
        dd_rym.change(budget_status, dd_rym, tbl_rep)
        with gr.Row():
            gr.Plot(value=net_chart())
            gr.Plot(value=share_chart())
print("✅ 三分頁組裝完成（記帳／轉帳與預算／報表）；下一格 launch")

In [ ]:
# [SKIP-TEST] 開帳！（報告時 app.launch(share=True) 讓全班手機操作）
app.launch(height=700)

# §9 測試總驗收（共同要求 8：≥8 assert、含 ≥2 個應該失敗）

In [ ]:
checks = []
def check(label, ok):
    checks.append((label, ok)); print(("✅" if ok else "❌"), label)

bal0 = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
check("記一筆支出成功", add_txn("皮夾現金", "飲料", 65).startswith("✅"))
bal1 = mcon.execute("SELECT balance FROM account WHERE aname='皮夾現金'").fetchone()[0]
check("餘額同步 -65", bal1 == bal0 - 65)
check("記一筆收入成功", add_txn("郵局帳戶", "獎學金", 5000).startswith("✅"))
b0 = mcon.execute("SELECT balance FROM account WHERE aname='悠遊卡'").fetchone()[0]
check("轉帳成功", transfer("郵局帳戶", "悠遊卡", 500).startswith("✅"))
b1 = mcon.execute("SELECT balance FROM account WHERE aname='悠遊卡'").fetchone()[0]
check("轉帳後票證 +500", b1 == b0 + 500)
check("設預算 UPSERT", set_budget("網購", DEMO_MONTH, 1234).startswith("✅") and
                       set_budget("網購", DEMO_MONTH, 999).startswith("✅"))
check("查流水有結果", len(list_txns()) > 0)

check("【應該失敗】負金額被擋", add_txn("皮夾現金", "晚餐", -10).startswith("⚠️"))
check("【應該失敗】現金刷爆被跨欄 CHECK 擋", add_txn("皮夾現金", "網購", 10**9).startswith("❌"))
check("【應該失敗】轉帳超額整筆回滾", transfer("皮夾現金", "悠遊卡", 10**9).startswith("❌"))
check("【應該失敗】收入分類不能設預算", set_budget("薪資家教", DEMO_MONTH, 100).startswith("⚠️"))
check("【應該失敗】非 ISO 交易日期被擋", add_txn("皮夾現金", "晚餐", 10, at="2026/11/15").startswith("❌"))
check("【應該失敗】非 ISO 預算月份被擋", set_budget("娛樂", "2026/11", 100).startswith("⚠️"))

# 終極對帳：每個帳戶的快照 = 流水結算（含轉帳方向）
n_bad = mcon.execute("""
    SELECT COUNT(*) FROM account a
    WHERE a.balance <> COALESCE((
        SELECT SUM(CASE c.kind WHEN '收入' THEN t.amount WHEN '支出' THEN -t.amount
                   WHEN '轉帳' THEN CASE WHEN c.cname = '轉入' THEN t.amount ELSE -t.amount END END)
        FROM txn t JOIN category c ON t.cat_id = c.cat_id
        WHERE t.account_id = a.account_id), 0)""").fetchone()[0]
check("全帳戶對帳：快照 = 流水結算", n_bad == 0)

assert all(ok for _, ok in checks), "有測試沒過！"
print(f"\n🎉 總驗收 {len(checks)} 項全數通過（其中「應該失敗」{sum('應該失敗' in n for n, _ in checks)} 項）")

# §10 AI 使用說明（共同要求 9——示範「怎麼寫」）

- **使用工具**：Claude、Gemini。
- **關鍵 prompt 摘錄**：
  1. 「個人記帳系統的需求如下（貼 §1）……請給 3NF DDL，並說明『轉帳』該怎麼建模、為什麼。」
  2. 「寫 add_txn()：金額一律正數、方向由分類決定、與餘額快照同交易更新；附 4 個 assert（兩個應該失敗）。」
  3. 「兩個裝置同時記帳，餘額會出什麼事？給我最小重現。」（反向質詢 → §5 的 lost update 劇本）
- **它哪裡不對、我怎麼修**：
  - 初版把轉帳建成「amount 允許負數」——與設計決策 1 衝突，改成「轉出／轉入成對正數」；
  - add_txn 的餘額更新它寫成 Python 先讀再寫——§5 證明會 lost update，改成 `balance = balance ± ?`；
  - 預算表它忘了 `UNIQUE(cat_id, ym)`——同月能設兩筆，用 UPSERT＋UNIQUE 收乾淨。
- **我如何確認正確**：§9 總驗收 14 項；「快照 vs 流水結算」全帳戶對帳；報表 1 的月結餘用 pandas
  重算抽查（雙引擎驗證）。
- **心得**：金流類系統的難點不在 CRUD，在**方向、一致性與併發**——AI 給的第一版幾乎都會踩其中一個，
  「應該失敗的測試」是最省力的照妖鏡。

# §11 十二分鐘 demo 腳本與備援（共同要求 10）

## 六步腳本（只選能串成故事的畫面）

| 時間 | 要說／要操作的事 | 成功訊號 |
|---|---|---|
| 0:00–1:00 | 一句需求＋「金額恆正」schema 決策 | 聽眾理解方向由分類決定 |
| 1:00–2:30 | 約束踩點、固定 seed 資料與餘額對帳 | 錯資料被擋，快照與流水一致 |
| 2:30–4:30 | 記一筆、轉一次帳；指出同一交易的動作 | 流水與帳戶餘額同步變化 |
| 4:30–7:00 | 先演 lost update，再跑 DB 端算術 | 壞版少扣一筆；好版兩筆都算進去 |
| 7:00–9:30 | 只挑結餘趨勢、預算達成＋索引證據 | 先回答問題，再展示 SCAN→SEARCH |
| 9:30–12:00 | UI 以示範日期記娛樂：500 元→逼近預算警示；700 元→超支；總驗收 | 操作、報表、測試形成閉環 |

## 三層備援

1. **A 案：live app**——報告前重跑全本，日期保留 `DEMO_TODAY`，照六步操作。
2. **B 案：notebook 輸出**——介面若失靈，展示先跑好的函數輸出、表格、圖與 assert；故事線不變。
3. **C 案：靜態證據**——網路或 Colab 中斷時，以事先截圖／短錄影呈現關鍵操作，並說出預期餘額與預算差額。

## 觀摩重點

- 快照（balance）與事實（txn 流水）並存時，對帳查詢就是安全網。
- lost update 和超賣（library 示範）是兩種不同競態；先辨認自己的題目屬於哪種。
- `DEMO_TODAY` 與當月預算相連，日期輸入一律是 `YYYY-MM-DD`，所以每次 demo 都能重現。
- UPSERT 讓「新增／修改預算」共用一個操作；分類的 `kind` 也讓報表一刀排除轉帳。
- 對照自己的題目：「餘額」可能是名額、庫存、點數或時數——找到它，這本就是攻略。

---
## 附錄：與指派題目的關係、重跑須知

- 本示範是「**交易流水＋庫存/餘額快照＋分析報表**」類題型的標竿；
  領域（個人記帳）**不在指派清單內**——結構學走，程式自己寫。
- 重跑：`執行階段 → 全部執行`（seed=7、模擬今日 `2026-11-15`，資料可重現）；`launch()` 在 Colab 內嵌介面。
- 想改造練手：把「訂閱服務」改成年繳攤提；幫信用卡加「月結日」邏輯；把預算警示接到記帳當下即時提醒。